# 잔해 방주 — 세계관 이미지 테스트 생성 (Colab 무료 GPU)

SDXL 애니 배경 모델로 `world_prompts.json`의 12장(힐링 스팟 6 + 세계관 6)을 뽑습니다.

**사용법**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 확인.
2. `런타임 → 모두 실행`. 첫 실행은 모델 다운로드로 5~8분 걸립니다.
3. 4번 셀에서 `world_prompts.json`을 올리라고 하면 프로젝트의 `data/world_prompts.json`을 선택합니다. 올리지 않으면 내장 기본값을 씁니다.
4. 끝나면 `relic_world_images.zip`이 자동 다운로드됩니다. 프로젝트의 `art_raw/world/`에 풀어 넣으세요.

장당 약 25초(T4). 무료 한도 안에서 12장은 넉넉합니다.

In [ ]:
#@title 1. 설치
!pip install -q diffusers==0.31.0 transformers accelerate safetensors
import torch, json, os, zipfile, random
from IPython.display import display
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — 런타임 유형을 T4 GPU로 바꾸세요')

In [ ]:
#@title 2. 모델 선택
MODEL = "cagliostrolab/animagine-xl-4.0"  #@param ["cagliostrolab/animagine-xl-4.0", "OnomaAIResearch/Illustrious-xl-early-release-v0", "stabilityai/stable-diffusion-xl-base-1.0"]
STEPS = 28  #@param {type:"integer"}
CFG = 6.0   #@param {type:"number"}
VARIANTS = 1  #@param {type:"integer"}  # 프롬프트당 장수 (시드 +1씩)
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
pipe = StableDiffusionXLPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, use_safetensors=True).to('cuda')
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing(); pipe.enable_attention_slicing()
print('loaded', MODEL)

In [ ]:
#@title 3. 프롬프트 불러오기 (업로드 또는 내장 기본값)
DEFAULT = {
 "_style": "anime background art, Makoto Shinkai and Studio Ghibli style, matte painting, lush overgrown greenery, dappled sunlight, saturated greens with warm orange accents, highly detailed, 8k, no text",
 "_negative": "blurry, text, watermark, signature, logo, deformed, low quality, extra limbs, distorted faces, jpeg artifacts",
 "_size": [832, 1216],
 "healing": [
  {"id": "spot_flooded_train", "seed": 88001, "prompt": "interior of an abandoned subway car flooded with crystal clear turquoise water, koi and goldfish swimming over the floor, pink and yellow flowers growing from the seats, vines on the hand straps, sunlight through the windows, serene"},
  {"id": "spot_goldfish_canal", "seed": 88002, "prompt": "an old covered city stream broken open, clear turquoise water flowing under mossy stone arch bridges, big orange goldfish visible underwater in cross-section, vines and trees over the bridge, small survivors walking on the stone path"},
  {"id": "spot_greenhouse_cafe", "seed": 88003, "prompt": "ruined glass arcade turned into a greenhouse, tall trees breaking through the iron and glass roof, abandoned cafe tables with orange awnings beside a flooded street of clear turquoise water, reflections, peaceful"},
  {"id": "spot_forest_train_door", "seed": 88004, "prompt": "inside an abandoned train car overgrown with moss and vines, open door revealing a sunlit forest with rails disappearing into the trees, orange seats, hand straps, warm light on the floor"},
  {"id": "spot_rooftop_garden", "seed": 88005, "prompt": "rooftop of a ruined department store turned into a wild garden, rainwater pond with lotus, deer drinking, distant overgrown city skyline, soft morning light"},
  {"id": "spot_lantern_river", "seed": 88006, "prompt": "night riverside in a ruined city, thousands of fireflies over dark water, old paper lanterns still hanging from a broken bridge, reflections, a lone survivor sitting quietly, warm and cool light"}
 ],
 "world": [
  {"id": "world_mall_interior", "seed": 88011, "prompt": "interior of a huge abandoned shopping mall, survivors camping between tall store shelves, tents and laundry lines, lanterns, plants growing through cracked floor, broken skylight letting in sunbeams, cozy survival"},
  {"id": "world_mall_exterior_dino", "seed": 88012, "prompt": "exterior of a giant overgrown shopping mall in a ruined city at dusk, a herd of triceratops grazing on a grassy parking lot, a tyrannosaurus silhouette in the mist behind broken buildings, warm lights in the mall windows"},
  {"id": "world_underground_warehouse", "seed": 88013, "prompt": "underground warehouse level of a mall turned into a home, concrete pillars, rooms built from shelving and tarps, warm lantern light, vines hanging from the ceiling, children playing, water dripping into a basin"},
  {"id": "world_forgotten_station", "seed": 88014, "prompt": "abandoned subway platform connected to a mall basement, tents on the platform, campfire, old train half in the tunnel, glowing green fungus in the dark tunnel, safety line and tiles"},
  {"id": "world_scavenging", "seed": 88015, "prompt": "two survivors pushing a shopping cart through an overgrown supermarket aisle, grabbing cans and instant noodles, sunlight through collapsed roof, a velociraptor shadow beyond the glass doors"},
  {"id": "world_barcode_reader", "seed": 88016, "prompt": "close-up of weathered hands holding an ancient handheld barcode scanner glowing faint gold, scanning a faded ramen packet on a stone shelf, dust in the lantern light, mysterious, sacred relic"}
 ]
}
P = DEFAULT
try:
    from google.colab import files
    print('world_prompts.json 을 올리세요 (취소하면 기본값 사용)')
    up = files.upload()
    for name, data in up.items():
        P = json.loads(data.decode('utf-8')); print('loaded', name)
except Exception as e:
    print('기본값 사용', e)
jobs = [(g, j) for g in ('healing', 'world') for j in P.get(g, [])]
print(len(jobs), 'prompts')

In [ ]:
#@title 4. 생성
os.makedirs('/content/out', exist_ok=True)
W, H = P.get('_size', [832, 1216])
QUALITY = 'masterpiece, best quality, very aesthetic, absurdres, scenery, ' if 'animagine' in MODEL or 'Illustrious' in MODEL else ''
for group, j in jobs:
    for v in range(VARIANTS):
        seed = int(j['seed']) + v
        g = torch.Generator('cuda').manual_seed(seed)
        prompt = QUALITY + j['prompt'] + ', ' + P['_style']
        img = pipe(prompt=prompt, negative_prompt=P['_negative'], width=W, height=H, num_inference_steps=STEPS, guidance_scale=CFG, generator=g).images[0]
        path = f"/content/out/{j['id']}{'' if v == 0 else '_v' + str(v)}.png"
        img.save(path); print('saved', path); display(img.resize((W // 3, H // 3)))

In [ ]:
#@title 5. 압축 후 다운로드
zp = '/content/relic_world_images.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('/content/out')): z.write(os.path.join('/content/out', f), f)
print(os.path.getsize(zp) // 1024, 'KB')
try:
    from google.colab import files; files.download(zp)
except Exception as e:
    print('수동 다운로드:', zp)

## (선택) 6. 우리 3D 렌더를 뼈대로 덧그리기 — ControlNet Depth
Blender에서 뽑은 **깊이 맵**(회색조 PNG)을 올리면, 격자 좌표를 지키면서 위 스타일로 방 타일을 다시 그립니다. 깊이 맵 출력은 `tools/blender_iso.py`에 `--depth` 옵션으로 추가 예정.

In [ ]:
#@title 6. ControlNet Depth로 방 타일 덧그리기 (깊이 PNG 업로드)
RUN_CONTROLNET = False  #@param {type:"boolean"}
if RUN_CONTROLNET:
    from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
    from PIL import Image
    cn = ControlNetModel.from_pretrained('diffusers/controlnet-depth-sdxl-1.0-small', torch_dtype=torch.float16)
    cpipe = StableDiffusionXLControlNetPipeline.from_pretrained(MODEL, controlnet=cn, torch_dtype=torch.float16, use_safetensors=True).to('cuda')
    cpipe.scheduler = EulerAncestralDiscreteScheduler.from_config(cpipe.scheduler.config)
    from google.colab import files
    up = files.upload()
    os.makedirs('/content/out_cn', exist_ok=True)
    for name, data in up.items():
        open('/tmp/' + name, 'wb').write(data)
        depth = Image.open('/tmp/' + name).convert('RGB').resize((1024, 1024))
        room = name.split('.')[0]
        prompt = QUALITY + f"isometric view of a cozy underground survival room ({room}), concrete walls covered in moss and vines, warm lantern, tents and supplies, " + P['_style']
        img = cpipe(prompt=prompt, negative_prompt=P['_negative'], image=depth, controlnet_conditioning_scale=0.65, num_inference_steps=STEPS, guidance_scale=CFG, generator=torch.Generator('cuda').manual_seed(880)).images[0]
        img.save(f'/content/out_cn/{room}.png'); display(img.resize((512, 512)))
    print('done → /content/out_cn')